# 03 — Análise Epidemiológica
**Objetivo:** Caracterizar o perfil das vítimas e correlacionar variáveis clínicas com desfecho.

> **⚠️ Nota sobre os dados:** Os dados individuais das 249 vítimas são **simulados** com base
> nas distribuições estatísticas reportadas por Brandão-Mello et al. (1991) e AIEA (1988).
> Os valores agregados (n total, óbitos, internações) são reais e documentados.

Análises:
- Perfil demográfico das vítimas
- Distribuição de doses por grupo de exposição
- Correlação dose × desfecho clínico
- Métricas epidemiológicas clássicas (CFR, taxa de ataque, incidência por grupo)
- Evolução temporal dos casos


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")

df = pd.read_csv("../data/processed/vitimas_clean.csv")
dosimetria = pd.read_csv("../data/processed/dosimetria_clean.csv")

print(f"Vítimas: {len(df)} | Internados: {df['internado'].sum()} | Óbitos: {df['obito'].sum()}")


## 3.1 — Perfil Demográfico

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Perfil Demográfico das Vítimas — Goiânia 1987', fontsize=13, fontweight='bold')

# Distribuição por sexo
sexo_counts = df['sexo'].value_counts()
axes[0].pie(sexo_counts, labels=['Masculino', 'Feminino'], autopct='%1.1f%%', colors=['#5B9BD5', '#ED7D31'])
axes[0].set_title('Distribuição por Sexo')

# Distribuição por idade
axes[1].hist(df['idade'], bins=15, color='#5B9BD5', edgecolor='white', linewidth=0.8)
axes[1].set_xlabel('Idade (anos)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição de Idades')

# Distribuição por grupo de exposição
grupo_counts = df['grupo_exposicao'].value_counts()
cores_grupo = {'leve': '#388E3C', 'moderado': '#F57C00', 'alto': '#D32F2F'}
cores = [cores_grupo[g] for g in grupo_counts.index]
axes[2].bar(grupo_counts.index, grupo_counts.values, color=cores, edgecolor='white')
axes[2].set_xlabel('Grupo de Exposição')
axes[2].set_ylabel('Número de Pacientes')
axes[2].set_title('Distribuição por Grupo de Exposição')
for i, v in enumerate(grupo_counts.values):
    axes[2].text(i, v + 1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/03_perfil_demografico.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.2 — Distribuição de Doses por Grupo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Preparar dados para boxplot
grupos_ordem = ['leve', 'moderado', 'alto']
n_por_grupo = {g: (df['grupo_exposicao'] == g).sum() for g in grupos_ordem}
data_plot = [df[df['grupo_exposicao'] == g]['dose_mSv'].values for g in grupos_ordem]

# CORRIGIDO: labels com n por grupo
labels_grupos = [
    f"Leve\n(n={n_por_grupo['leve']})",
    f"Moderado\n(n={n_por_grupo['moderado']})",
    f"Alto\n(n={n_por_grupo['alto']})"
]

bp = axes[0].boxplot(data_plot, labels=labels_grupos, patch_artist=True)
cores = ['#388E3C', '#F57C00', '#D32F2F']
for patch, cor in zip(bp['boxes'], cores):
    patch.set_facecolor(cor)
    patch.set_alpha(0.7)
axes[0].set_yscale('log')
axes[0].set_ylabel('Dose (mSv) — escala log')
axes[0].set_title('Distribuição de Doses por Grupo')

# Histograma de doses (escala log)
axes[1].hist(np.log10(df['dose_mSv'] + 1), bins=30, color='#5B9BD5', edgecolor='white')
axes[1].set_xlabel('log10(Dose mSv + 1)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Histograma de Doses (escala log10)')

# Linhas de limiar
for limiar, nome, cor in [(np.log10(101), '100 mSv', '#F57C00'), (np.log10(1001), '1000 mSv', '#D32F2F')]:
    axes[1].axvline(limiar, color=cor, linestyle='--', alpha=0.7, label=nome)
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/03_distribuicao_doses.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.3 — Métricas Epidemiológicas Clássicas

Métricas baseadas nos dados reais do acidente (AIEA, 1988).
- **População triada total:** 112.000 pessoas
- **Contaminados diretos (sintomáticos):** 249
- **Hospitalizados:** 57
- **Óbitos (síndrome aguda de radiação):** 4


In [ ]:
# Populacao triada total (dado real do AIEA)
n_triados = 112000
n_total = len(df)
n_internados = int(df['internado'].sum())
n_obitos = int(df['obito'].sum())

# ---- Taxa de Ataque (Attack Rate) ----
# Proporção de pessoas triadas que tiveram contaminação relevante
ar = n_total / n_triados

# ---- Case Fatality Rate (CFR) ----
# Proporção de óbitos entre hospitalizados (casos graves)
cfr = n_obitos / n_internados

# ---- Incidência de hospitalização por grupo de exposição ----
print('=== MÉTRICAS EPIDEMIOLÓGICAS CLÁSSICAS ===')
print(f'\nPopulação triada total:  {n_triados:,}')
print(f'Contaminados diretos:    {n_total}')
print(f'Hospitalizados:          {n_internados}')
print(f'Óbitos (SAR):            {n_obitos}')

print(f'\nTaxa de Ataque (triados → contaminados):  {ar:.4%}')
print(f'Case Fatality Rate (hospitalizados):      {cfr:.4%}')
print(f'CFR geral (contaminados → óbito):         {n_obitos/n_total:.4%}')

print('\n=== INCIDÊNCIA DE HOSPITALIZAÇÃO POR GRUPO ===')
for grupo in ['leve', 'moderado', 'alto']:
    sub = df[df['grupo_exposicao'] == grupo]
    n_g = len(sub)
    int_g = int(sub['infer'nado'].sum())
    obt_g = int(sub['obito'].sum())
    ir = int_g / n_g
    print(f'  {grupo.capitalize():10s}: n={n_g:3d} | hospitalizados={int_g:2d} | óbitos={obt_g} | IR={ir:.2%}')


## 3.4 — Correlação Dose × Desfecho

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Dose vs. Desfecho Clínico', fontsize=13, fontweight='bold')

# Scatter dose vs. dias_desfecho, colorido por internado
cores_scatter = df['internado'].map({0: '#90CAF9', 1: '#D32F2F'})
axes[0].scatter(df['dose_mSv'], df['dias_desfecho'], c=cores_scatter, alpha=0.5, s=20)
axes[0].set_xscale('log')
axes[0].set_xlabel('Dose (mSv) — escala log')
axes[0].set_ylabel('Dias até desfecho')
axes[0].set_title('Dose × Tempo ao Desfecho')
patch_nao = mpatches.Patch(color='#90CAF9', label='Não internado')
patch_sim = mpatches.Patch(color='#D32F2F', label='Internado')
axes[0].legend(handles=[patch_nao, patch_sim])

# Taxa de internação por quartil de dose
df['quartil_dose'] = pd.qcut(df['dose_mSv'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
taxa_int = df.groupby('quartil_dose')['internado'].mean()
axes[1].bar(taxa_int.index, taxa_int.values, color='#5B9BD5', edgecolor='white')
axes[1].set_xlabel('Quartil de Dose')
axes[1].set_ylabel('Taxa de Hospitalização')
axes[1].set_title('Taxa de Hospitalização por Quartil de Dose')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.savefig('../reports/figures/03_correlacao_dose_desfecho.png', dpi=150, bbox_inches='tight')
plt.show()
